# Ders 3A — 12 Model Araması

Seçilen gatekeeper feature setleriyle 12 klasik/boosting model karşılaştırılır.

## 1. Paket kontrolü

Bu scriptte gerçekten kullanılacak paketler kontrol edilir. Gereksiz paket import edilmez.

`XGBoost` ve `CatBoost` bu dosyada model listesine eklendiği için burada ayrıca kontrol edilir ve eksikse kurulur.

In [ ]:
import sys
# Mevcut Python ortamında pip komutunu çalıştırmak için kullanılır.
import subprocess
# Eksik paketleri notebook içinden kurmak için kullanılır.
import importlib.util
# Bir paketin kurulu olup olmadığını kontrol etmek için kullanılır.
import importlib
# Kurulumdan sonra import cache bilgisini yenilemek için kullanılır.

def install_if_missing(import_name, pip_name=None):
    """Eksik paket varsa kurar; paket zaten varsa işlem yapmaz."""
    pip_name = pip_name or import_name
    # pip adı verilmezse import adı paket adı olarak kullanılır.
    if importlib.util.find_spec(import_name) is None:
        # Paket kurulu değilse pip kurulumu yapılır.
        print(f"[INSTALL] {pip_name}")
        # Hangi paketin kurulacağı ekrana yazdırılır.
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        # Paket sessiz modda kurulur.
        importlib.invalidate_caches()
        # Kurulumdan sonra Python import cache bilgisi yenilenir.
    else:
        print(f"[OK] {import_name}")
        # Paket zaten kuruluysa tekrar kurulmaz.

required_packages = [
    ("pandas", "pandas"),        # CSV okuma ve tablo işlemleri için kullanılır.
    ("numpy", "numpy"),          # Sayısal matris ve vektör işlemleri için kullanılır.
    ("matplotlib", "matplotlib"),# Grafik çizimleri için kullanılır.
    ("sklearn", "scikit-learn"), # Makine öğrenmesi modelleri ve metrikleri için kullanılır.
    ("xgboost", "xgboost"),      # XGBoost modelini kullanmak için gereklidir.
    ("catboost", "catboost"),    # CatBoost modelini kullanmak için gereklidir.
]
# Bu scriptte gerçekten kullanılan paketler listelenir.

for import_name, pip_name in required_packages:
    # Paketler tek tek kontrol edilir.
    install_if_missing(import_name, pip_name)
    # Eksik paket varsa kurulur.

print("Paket kontrolü tamamlandı.")
# Paket kontrolünün bittiği yazdırılır.


## 2. Importlar ve genel ayarlar

Bu hücrede yalnızca bu scriptte kullanılacak paketler ve sabitler çağırılır.

In [ ]:
from pathlib import Path
# Dosya ve klasör yollarını güvenli şekilde yönetmek için kullanılır.
import warnings
# Gereksiz uyarıları kontrol etmek için kullanılır.
warnings.filterwarnings("ignore")
# Notebook çıktısını sade tutmak için uyarılar gizlenir.

import numpy as np
# Feature matrisleri, indeksleme ve skor sıralama için kullanılır.
import pandas as pd
# Feature CSV dosyalarını okumak ve sonuç tablolarını kaydetmek için kullanılır.
import matplotlib.pyplot as plt
# Model karşılaştırma bar chartlarını çizmek için kullanılır.

from sklearn.model_selection import train_test_split
# Train/test ayrımı yapmak için kullanılır.
from sklearn.metrics import accuracy_score, balanced_accuracy_score, average_precision_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
# 10 modelin performansını ölçmek için gerekli metrikler çağırılır.
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier, GradientBoostingClassifier
# Ağaç tabanlı klasik modeller çağırılır.
from sklearn.pipeline import Pipeline
# Ölçekleme isteyen modeller için preprocessing + model akışı kurar.
from sklearn.preprocessing import StandardScaler
# SVM/kNN/logistic regression için ölçekleme kullanılır.
from sklearn.impute import SimpleImputer
# Eksik değerleri train setin median değerleriyle doldurmak için kullanılır.
from sklearn.feature_selection import VarianceThreshold
# Pipeline içinde sabit/değişmeyen feature kolonlarını kaldırmak için kullanılır.
from sklearn.linear_model import LogisticRegression
# Logistic regression modeli çağırılır.
from sklearn.neighbors import KNeighborsClassifier
# kNN modeli çağırılır.
from sklearn.svm import SVC, LinearSVC
# Linear ve RBF SVM modelleri çağırılır.
from sklearn.calibration import CalibratedClassifierCV
# LinearSVC çıktısını olasılık benzeri skora dönüştürmek için kullanılır.
from sklearn.naive_bayes import GaussianNB
# Gaussian Naive Bayes modeli çağırılır.
from sklearn.tree import DecisionTreeClassifier
# Decision Tree modeli çağırılır.
from xgboost import XGBClassifier
# XGBoost gradient boosting tabanlı güçlü bir model olarak eklenir.
from catboost import CatBoostClassifier
# CatBoost gradient boosting tabanlı güçlü bir model olarak eklenir.


DATASETS = {
    # İki veri setinin kısa isimleri ve GitHub raw feature dosyaları burada tanımlanır.
    "ERa_BLA_assay": {
        "short_name": "ERα BLA",
        "model_prefix": "model_ERa_BLA",
        "feature_url": "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/molfest_outputs/01_feature_store/model_ERa_BLA_features.csv",
        "raw_url": "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/Train_2_1_A14_A17_ERa_BLA_agonist_antagonist.csv",
    },
    "ERa_LUC_VM7_assay": {
        "short_name": "ERα LUC VM7",
        "model_prefix": "model_ERa_LUC_VM7",
        "feature_url": "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/molfest_outputs/01_feature_store/model_ERa_LUC_VM7_features.csv",
        "raw_url": "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/Train_2_2_A15_A18_ERa_LUC_VM7_agonist_antagonist.csv",
    },
}

RANDOM_STATE = 42
# Train/test ayrımı ve modeller için sabit random seed belirlenir.
TEST_SIZE = 0.20
# Test set oranı %20 olarak belirlenir.

OUTPUT_ROOT = Path("molfest_outputs")
# Bu scriptin çıktılarının kaydedileceği ana klasör belirlenir.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
# Çıktı klasörü yoksa oluşturulur.

print("Ayarlar hazır.")
# Ayar hücresinin tamamlandığı yazdırılır.
print("Feature CSV klasörü:", "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/molfest_outputs/01_feature_store")
# Hazır feature dosyalarının okunacağı GitHub raw klasörü gösterilir.

## 3. Yardımcı fonksiyonlar

Bu fonksiyonlar notebook içinde görünür durumdadır ve sadece bu scriptte ihtiyaç duyulan işlemleri kapsar.

In [ ]:
def show_table(df, n=50, title=None):
    """Sonuç tablosunu Colab'da tablo olarak, terminalde metin olarak gösterir."""
    if title:
        # Tablo başlığı varsa önce başlık yazdırılır.
        print("\n" + title)
    try:
        # Colab/Jupyter ortamında display daha okunabilir tablo üretir.
        display(df.head(n))
    except NameError:
        # Terminalde display yoksa tablo metin olarak yazdırılır.
        print(df.head(n).to_string(index=False))


def load_feature_table(dataset_key):
    """Seçilen veri setinin hazır feature CSV dosyasını GitHub raw linkinden okur."""
    info = DATASETS[dataset_key]
    # Veri setine ait URL ve kısa isim bilgisi alınır.
    df = pd.read_csv(info["feature_url"])
    # Feature CSV doğrudan GitHub raw linkinden okunur.
    print(f"\n{dataset_key} okundu: {df.shape[0]} satır, {df.shape[1]} kolon")
    # Okunan tablonun boyutu ekrana yazdırılır.
    return df
    # Okunan tablo modelleme adımlarına gönderilir.


def feature_columns(df, feature_set):
    """İstenen feature ailesine ait kolonları seçer."""
    prefix_map = {
        # Feature set adları CSV kolon prefixleriyle eşleştirilir.
        "morgan": ["Morgan_"],
        "maccs": ["MACCS_"],
        "rdkit": ["RDKit_"],
        "avalon": ["Avalon_"],
        "maccs_morgan": ["MACCS_", "Morgan_"],
        "maccs_rdkit": ["MACCS_", "RDKit_"],
        "morgan_rdkit": ["Morgan_", "RDKit_"],
        "all_available": ["Morgan_", "MACCS_", "RDKit_", "Avalon_"],
    }
    prefixes = prefix_map[feature_set]
    # İstenen feature setinin kolon prefixleri alınır.
    cols = [c for c in df.columns if any(c.startswith(p) for p in prefixes)]
    # Prefix ile eşleşen feature kolonları seçilir.
    if not cols:
        # Hiç kolon bulunmazsa erken ve anlaşılır hata verilir.
        raise ValueError(f"{feature_set} için feature kolonu bulunamadı.")
    return cols
    # Seçilen feature kolon isimleri döndürülür.


def make_xy(df, cols):
    """Feature kolonlarından X matrisi ve Target kolonundan y vektörü üretir."""
    X = (
        df[cols]
        # Sadece seçilen feature kolonları alınır.
        .apply(pd.to_numeric, errors="coerce")
        # Sayısal olmayan değerler güvenli şekilde NaN yapılır.
        .replace([np.inf, -np.inf], np.nan)
        # Sonsuz değerler NaN'a çevrilir.
        .to_numpy(dtype=np.float32)
        # sklearn modelleri için numpy matrise çevrilir.
    )
    y = df["Target"].astype(int).to_numpy()
    # Binary target kolonu integer numpy vektörüne çevrilir.
    return X, y
    # Modelleme için X ve y döndürülür.


def split_data(df, cols):
    """Sınıf oranını koruyarak train/test ayrımı yapar."""
    X, y = make_xy(df, cols)
    # Seçilen featurelardan X ve y hazırlanır.
    return train_test_split(X, y, df, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
    # Stratify ile sınıf oranı train/test içinde korunur.


def impute_train_test(X_train, X_test):
    """Eksik değerleri sadece train setten öğrenilen median değerlerle doldurur."""
    imputer = SimpleImputer(strategy="median")
    # SimpleImputer eksik değerleri her feature'ın train set median değeriyle doldurur.
    X_train_imputed = imputer.fit_transform(X_train)
    # Imputer sadece train set üzerinde öğrenilir; böylece test bilgisinin train'e sızması engellenir.
    X_test_imputed = imputer.transform(X_test)
    # Test set aynı imputer ile dönüştürülür.
    return X_train_imputed, X_test_imputed
    # Model eğitiminde kullanılacak temiz train/test matrisleri döndürülür.


def class1_score(model, X):
    """Modelden class 1 için skor veya olasılık üretir."""
    if hasattr(model, "predict_proba"):
        # Model olasılık üretebiliyorsa class 1 olasılığı alınır.
        score = model.predict_proba(X)[:, 1]
        # Class 1 olasılıkları alınır.
    elif hasattr(model, "decision_function"):
        # Olasılık yoksa karar fonksiyonu skoru kullanılır.
        score = model.decision_function(X)
        # Karar fonksiyonu skoru alınır.
    else:
        score = model.predict(X).astype(float)
        # Son seçenek olarak sınıf tahmini skor gibi kullanılır.

    score = np.asarray(score, dtype=float)
    # Skorlar numpy array formatına çevrilir.
    score = np.nan_to_num(score, nan=0.5, posinf=1.0, neginf=0.0)
    # Bazı modeller nadiren NaN/inf skor üretebilir; ROC hesabı bozulmasın diye güvenli aralığa çekilir.
    return score
    # Temizlenmiş class 1 skoru döndürülür.


def metric_dict(y_true, y_pred, y_score):
    """Test tahminlerinden temel sınıflandırma metriklerini hesaplar."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    # Confusion matrix değerleri ayrı ayrı alınır.
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    # Specificity = TN / (TN + FP) olarak hesaplanır.
    return {
        "ROC": roc_auc_score(y_true, y_score),
        # ROC-AUC, pozitif ve negatif sınıf ayrım gücünü ölçer.
        "AP": average_precision_score(y_true, y_score),
        # AP, precision-recall eğrisi altındaki ortalama performanstır.
        "F1": f1_score(y_true, y_pred, zero_division=0),
        # F1, precision ve recall dengesini özetler.
        "Accuracy": accuracy_score(y_true, y_pred),
        # Accuracy, toplam doğru sınıflandırma oranıdır.
        "BalancedAccuracy": balanced_accuracy_score(y_true, y_pred),
        # Balanced accuracy, sınıf dengesizliğinde daha adil bir doğruluk ölçüsüdür.
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        # Recall = TP / (TP + FN) olarak hesaplanır.
        "Specificity": specificity,
        # Specificity = TN / (TN + FP) olarak hesaplanır.
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        # Precision = TP / (TP + FP) olarak hesaplanır.
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    }


def plot_metric_bar(df, label_col, metric, title, out_file=None, top_n=None):
    """Performans sonuçlarını yatay bar chart olarak çizer."""
    plot_df = df.sort_values(metric, ascending=False).copy()
    # En iyi sonuçlar üstte olacak şekilde metrik değerine göre sıralanır.
    if top_n is not None:
        # Çok uzun tablolar için ilk n sonuç çizilebilir.
        plot_df = plot_df.head(top_n)
    plot_df = plot_df.sort_values(metric, ascending=True)
    # Yatay bar chartta en iyi değer üstte görünsün diye çizim öncesi ters sıralanır.

    plt.figure(figsize=(9, max(4, 0.35 * len(plot_df))))
    # Grafik boyutu satır sayısına göre ayarlanır.
    plt.barh(plot_df[label_col].astype(str), plot_df[metric])
    # Her model/feature set için metrik değeri yatay bar olarak çizilir.
    plt.xlabel(metric)
    # X eksenine hangi metrik çizildiği yazılır.
    if metric == "ROC":
        # ROC-AUC farkları daha net görünsün diye ROC grafikleri 0.60'tan başlatılır.
        plt.xlim(left=0.60)
    plt.title(title)
    # Grafiğe açıklayıcı başlık eklenir.
    plt.tight_layout()
    # Etiketlerin kesilmemesi için yerleşim düzenlenir.
    if out_file:
        # Dosya yolu verilmişse grafik kaydedilir.
        Path(out_file).parent.mkdir(parents=True, exist_ok=True)
        # Kayıt klasörü yoksa oluşturulur.
        plt.savefig(out_file, dpi=300, bbox_inches="tight")
        # Grafik yüksek çözünürlüklü PNG olarak kaydedilir.
    plt.show()
    # Grafik ekranda gösterilir.


def add_gain(current_df, previous_df, current_name, previous_name):
    """Bir önceki adıma göre ROC/AP/F1 farkını ve yüzde değişimi hesaplar."""
    rows = []
    # Hesaplanan karşılaştırma satırları burada biriktirilir.
    for _, current in current_df.iterrows():
        # Mevcut adımın her veri seti sonucu tek tek dolaşılır.
        dataset = current["Dataset"]
        # Karşılaştırılacak veri seti adı alınır.
        prev = previous_df[previous_df["Dataset"] == dataset].sort_values("ROC", ascending=False).iloc[0]
        # Aynı veri setinin önceki adımdaki en iyi sonucu alınır.
        row = current.to_dict()
        # Mevcut sonuç satırı sözlüğe çevrilir.
        row["CurrentStep"] = current_name
        # Mevcut adım adı eklenir.
        row["ComparedWith"] = previous_name
        # Karşılaştırılan önceki adım adı eklenir.
        row["Previous_ROC"] = prev["ROC"]
        # Önceki ROC değeri tabloya eklenir.
        row["ROC_Delta"] = current["ROC"] - prev["ROC"]
        # Mutlak ROC farkı hesaplanır.
        row["ROC_Gain_%"] = 100 * (current["ROC"] - prev["ROC"]) / abs(prev["ROC"]) if abs(prev["ROC"]) > 1e-12 else np.nan
        # ROC yüzde değişimi hesaplanır.
        row["Previous_AP"] = prev["AP"]
        # Önceki AP değeri tabloya eklenir.
        row["AP_Delta"] = current["AP"] - prev["AP"]
        # AP farkı hesaplanır.
        row["AP_Gain_%"] = 100 * (current["AP"] - prev["AP"]) / abs(prev["AP"]) if abs(prev["AP"]) > 1e-12 else np.nan
        # AP yüzde değişimi hesaplanır.
        row["Previous_F1"] = prev["F1"]
        # Önceki F1 değeri tabloya eklenir.
        row["F1_Delta"] = current["F1"] - prev["F1"]
        # F1 farkı hesaplanır.
        row["F1_Gain_%"] = 100 * (current["F1"] - prev["F1"]) / abs(prev["F1"]) if abs(prev["F1"]) > 1e-12 else np.nan
        # F1 yüzde değişimi hesaplanır.
        rows.append(row)
        # Karşılaştırma satırı listeye eklenir.
    return pd.DataFrame(rows)
    # Tüm karşılaştırmalar tablo olarak döndürülür.


def make_random_forest():
    """Baseline ve gatekeeper için RandomForest modeli kurar."""
    return RandomForestClassifier(
        n_estimators=300,
        # 300 ağaç, eğitim süresi ve stabilite arasında dengeli bir seçimdir.
        max_features="sqrt",
        # Her bölünmede featureların karekökü kadar aday denenir.
        class_weight="balanced_subsample",
        # Sınıf dengesizliğini her bootstrap örneğinde dengelemeye çalışır.
        random_state=RANDOM_STATE,
        # Tekrarlanabilirlik sağlar.
        n_jobs=-1,
        # Mevcut tüm işlemciler kullanılır.
    )


def make_model(model_type):
    """Model adından sklearn model nesnesi üretir."""
    if model_type == "RandomForest":
        # RandomForest güçlü ve stabil bir tree-based gatekeeper modelidir.
        return RandomForestClassifier(n_estimators=300, max_features="sqrt", class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1)
    if model_type == "ExtraTrees":
        # ExtraTrees, bölünmeleri daha rastgele seçen güçlü bir ağaç ensemble modelidir.
        return ExtraTreesClassifier(n_estimators=300, max_features="sqrt", class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
    if model_type == "HistGradientBoosting":
        # HistGradientBoosting, histogram tabanlı hızlı boosting modelidir.
        return HistGradientBoostingClassifier(max_iter=120, learning_rate=0.08, random_state=RANDOM_STATE)
    if model_type == "LogisticRegression":
        # Logistic regression ölçekleme istediği için pipeline içinde kullanılır.
        return Pipeline([
            ("variance", VarianceThreshold(0.0)),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear")),
        ])
    if model_type == "kNN":
        # kNN mesafe tabanlı olduğu için ölçekleme ile birlikte kullanılır.
        return Pipeline([
            ("variance", VarianceThreshold(0.0)),
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier(n_neighbors=7, weights="distance")),
        ])
    if model_type == "LinearSVM":
        # LinearSVM karar skoru üretir; kalibrasyonla olasılık benzeri skor alınır.
        return Pipeline([
            ("variance", VarianceThreshold(0.0)),
            ("scaler", StandardScaler()),
            ("model", CalibratedClassifierCV(LinearSVC(class_weight="balanced", random_state=RANDOM_STATE), cv=3)),
        ])
    if model_type == "RBF_SVM":
        # RBF kernel SVM doğrusal olmayan sınırları yakalayabilir.
        return Pipeline([
            ("variance", VarianceThreshold(0.0)),
            ("scaler", StandardScaler()),
            ("model", SVC(C=3.0, kernel="rbf", gamma="scale", class_weight="balanced", probability=True, random_state=RANDOM_STATE)),
        ])
    if model_type == "GradientBoosting":
        # Klasik gradient boosting modeli karşılaştırma için eklenir.
        return GradientBoostingClassifier(random_state=RANDOM_STATE)
    if model_type == "GaussianNB":
        # Basit ve hızlı bir Naive Bayes karşılaştırma modelidir.
        return GaussianNB()
    if model_type == "DecisionTree":
        # Tek karar ağacı, ensemble modellere göre daha basit bir referanstır.
        return DecisionTreeClassifier(max_depth=8, class_weight="balanced", random_state=RANDOM_STATE)
    if model_type == "XGBoost":
        # XGBoost güçlü bir gradient boosting modelidir.
        return XGBClassifier(
            n_estimators=300,
            # 300 boosting ağacı kullanılır.
            max_depth=4,
            # Ağaç derinliği orta seviyede tutulur.
            learning_rate=0.05,
            # Daha kontrollü öğrenme için düşük öğrenme oranı kullanılır.
            subsample=0.9,
            # Her iterasyonda verinin %90'ı kullanılır.
            colsample_bytree=0.9,
            # Her ağaçta featureların %90'ı kullanılır.
            eval_metric="logloss",
            # XGBoost için değerlendirme metriği tanımlanır.
            random_state=RANDOM_STATE,
            # Tekrarlanabilirlik sağlar.
            n_jobs=-1,
            # Mevcut işlemciler kullanılır.
        )
    if model_type == "CatBoost":
        # CatBoost güçlü bir gradient boosting modelidir.
        return CatBoostClassifier(
            iterations=300,
            # 300 boosting iterasyonu kullanılır.
            depth=4,
            # Ağaç derinliği orta seviyede tutulur.
            learning_rate=0.05,
            # Daha kontrollü öğrenme için düşük öğrenme oranı kullanılır.
            loss_function="Logloss",
            # Binary classification için logloss kullanılır.
            random_seed=RANDOM_STATE,
            # Tekrarlanabilirlik sağlar.
            verbose=False,
            # Eğitim sırasında uzun log basılması engellenir.
        )
    raise ValueError(f"Bilinmeyen model tipi: {model_type}")


## 4. Step 3 gatekeeper feature setleri

Step 4'te anlamlı bir artış olmadığı için bu dosyada feature selection yapılmaz.

10 model araması doğrudan Step 3'te seçilen gatekeeper feature setleri üzerinden yapılır.

- `ERa_BLA_assay`: `rdkit`
- `ERa_LUC_VM7_assay`: `all_available`


In [ ]:
forced_feature_sets = {
    "ERa_BLA_assay": "rdkit",
    "ERa_LUC_VM7_assay": "all_available",
}
# Step 3 sonucuna göre devam edilecek feature setler sabitlenir.
# Bu dosyada feature selection yapılmaz.

gatekeeper_rows = []
# Her veri seti için gatekeeper RF sonucu burada tutulur.

for dataset_key in DATASETS:
    # Her veri seti sırayla işlenir.
    df = load_feature_table(dataset_key)
    # Hazır feature CSV dosyası okunur.

    selected_feature_set = forced_feature_sets[dataset_key]
    # Bu veri seti için Step 3'ten taşınan feature set alınır.
    selected_features = feature_columns(df, selected_feature_set)
    # Seçilen feature setin kolonları alınır.

    X_train, X_test, y_train, y_test, df_train, df_test = split_data(df, selected_features)
    # Seçilen feature setiyle train/test ayrımı yapılır.

    X_train, X_test = impute_train_test(X_train, X_test)
    # Eksik değerler SimpleImputer ile doldurulur.
    # Imputer median değerleri sadece train setten öğrenir ve sonra test sete uygular.

    gatekeeper_model = make_random_forest()
    # Gatekeeper model olarak RandomForest kurulur.
    gatekeeper_model.fit(X_train, y_train)
    # Gatekeeper model train set üzerinde eğitilir.

    y_pred = gatekeeper_model.predict(X_test)
    # Test seti için sınıf tahmini alınır.
    y_score = class1_score(gatekeeper_model, X_test)
    # ROC/AP için class 1 skoru alınır.

    metrics = metric_dict(y_test, y_pred, y_score)
    # Test performans metrikleri hesaplanır.
    metrics.update({
        "Dataset": dataset_key,
        "Comparison": "Gatekeeper",
        "Step": "03_gatekeeper_feature_set",
        "ModelType": "RandomForest",
        "FeatureSet": selected_feature_set,
        "SelectionMethod": "none",
        "K": len(selected_features),
        "FeatureCount": len(selected_features),
        "SelectedFeatures": selected_features,
    })
    # Sonuç satırına gatekeeper feature set bilgileri eklenir.

    gatekeeper_rows.append(metrics)
    # Gatekeeper sonucu listeye eklenir.

gatekeeper_df = pd.DataFrame(gatekeeper_rows)
# Gatekeeper sonuçları tabloya çevrilir.

show_table(
    gatekeeper_df[["Dataset", "FeatureSet", "ModelType", "FeatureCount", "ROC", "AP", "F1"]],
    title="10 model araması için kullanılacak Step 3 gatekeeper feature setleri"
)
# 10 model aramasında kullanılacak feature setler ekranda gösterilir.


## 5. Gatekeeper feature setleriyle 12 model eğitimi

Bu hücrede her model için `model.fit`, `model.predict`, `class1_score` ve metrik hesaplama açıkça görünür.

Feature selection yapılmaz. Amaç, Step 3 gatekeeper RF sonucunu aynı feature set üzerinde eğitilen 12 klasik/boosting model ile karşılaştırmaktır.

Bu karşılaştırmada iki veri setinde de ROC-AUC temelli en iyi/taşınacak model RandomForest olduğu için sonraki adımlarda yalnızca RandomForest ile devam edilir.

Eksik değerler `SimpleImputer(strategy="median")` ile doldurulur. XGBoost ve CatBoost da model listesine eklenmiştir.


In [ ]:
lesson_out = OUTPUT_ROOT / "05_train_12_models"
# 12 model karşılaştırması çıktıları için klasör belirlenir.
lesson_out.mkdir(parents=True, exist_ok=True)
# Çıktı klasörü yoksa oluşturulur.

model_types = [
    "LogisticRegression",
    "kNN",
    "LinearSVM",
    "RBF_SVM",
    "RandomForest",
    "ExtraTrees",
    "GradientBoosting",
    "HistGradientBoosting",
    "GaussianNB",
    "DecisionTree",
    "XGBoost",
    "CatBoost",
]
# Karşılaştırılacak 12 model listelenir.

all_model_rows = []
# Tüm model sonuçları burada tutulur.
best_model_rows = []
# Her veri seti için en iyi model sonucu burada tutulur.
next_step_model_rows = []
# Sonraki adıma taşınacak tek model olan RandomForest sonuçları burada tutulur.
comparison_rows = []
# Gatekeeper vs en iyi model karşılaştırma satırları burada tutulur.

for dataset_key in DATASETS:
    # Her veri seti sırayla işlenir.
    df = load_feature_table(dataset_key)
    # Hazır feature CSV dosyası okunur.

    gatekeeper_row = gatekeeper_df[gatekeeper_df["Dataset"] == dataset_key].iloc[0]
    # Bu veri seti için Step 3 gatekeeper sonucu alınır.
    selected_features = list(gatekeeper_row["SelectedFeatures"])
    # Step 3 gatekeeper feature setindeki feature isimleri alınır.

    X_train, X_test, y_train, y_test, df_train, df_test = split_data(df, selected_features)
    # Seçilen featurelarla train/test ayrımı yapılır.

    X_train, X_test = impute_train_test(X_train, X_test)
    # Eksik değerler SimpleImputer ile doldurulur.
    # Imputer median değerleri sadece train setten öğrenir ve sonra test sete uygular.

    dataset_rows = []
    # Bu veri setindeki model sonuçları burada tutulur.

    for model_type in model_types:
        # 12 model sırayla denenir.
        model = make_model(model_type)
        # Model nesnesi oluşturulur.
        model.fit(X_train, y_train)
        # Model train set üzerinde eğitilir.

        y_pred = model.predict(X_test)
        # Test seti için sınıf tahmini alınır.
        y_score = class1_score(model, X_test)
        # ROC/AP için class 1 skoru alınır.

        metrics = metric_dict(y_test, y_pred, y_score)
        # Test performans metrikleri hesaplanır.
        metrics.update({
            "Dataset": dataset_key,
            "Step": "05_12_models",
            "ModelType": model_type,
            "FeatureSet": gatekeeper_row["FeatureSet"],
            "SelectionMethod": "none",
            "K": gatekeeper_row["K"],
            "FeatureCount": len(selected_features),
        })
        # Sonuç satırına model ve feature bilgileri eklenir.

        dataset_rows.append(metrics)
        # Bu veri seti sonuçlarına eklenir.
        all_model_rows.append(metrics)
        # Genel sonuç listesine eklenir.

    dataset_df = pd.DataFrame(dataset_rows).sort_values("ROC", ascending=False)
    # Veri seti sonuçları ROC'a göre sıralanır.
    dataset_df.to_csv(lesson_out / f"{dataset_key}_12_model_metrics.csv", index=False)
    # Veri setine ait model karşılaştırma tablosu CSV olarak kaydedilir.

    plot_metric_bar(
        dataset_df,
        label_col="ModelType",
        metric="ROC",
        title=f"{dataset_key} — 12 model ROC karşılaştırması",
        out_file=lesson_out / f"{dataset_key}_12_model_roc.png",
    )
    # 12 modelin ROC değerleri bar chart olarak çizilir.

    show_table(dataset_df, title=f"{dataset_key} — 12 model karşılaştırması")
    # 12 model sonuç tablosu ekranda gösterilir.

    best_model = dataset_df.iloc[0].to_dict()
    # ROC-AUC değerine göre en iyi model seçilir.
    best_model_rows.append(best_model)
    # En iyi genel model sonucu saklanır.

    rf_for_next_step = dataset_df[dataset_df["ModelType"] == "RandomForest"].copy()
    # İki veri setinde de en iyi/taşınacak model RandomForest olduğu için sonraki adımda yalnızca RF saklanır.
    rf_for_next_step["NextStepDecision"] = "Continue with RandomForest"
    # Sonraki adıma hangi modelin taşındığı açıkça yazılır.
    next_step_model_rows.append(rf_for_next_step)
    # RandomForest sonucu sonraki adım tablosuna eklenir.

    reference_row = {
        "Dataset": dataset_key,
        "Comparison": "Gatekeeper RF",
        "ModelType": "RandomForest",
        "FeatureSet": gatekeeper_row["FeatureSet"],
        "SelectionMethod": "none",
        "K": int(gatekeeper_row["K"]),
        "FeatureCount": int(gatekeeper_row["FeatureCount"]),
        "ROC": gatekeeper_row["ROC"],
        "AP": gatekeeper_row["AP"],
        "F1": gatekeeper_row["F1"],
        "ROC_Delta_vs_Gatekeeper": 0.0,
        "ROC_Gain_%": 0.0,
        "AP_Delta_vs_Gatekeeper": 0.0,
        "AP_Gain_%": 0.0,
        "F1_Delta_vs_Gatekeeper": 0.0,
        "F1_Gain_%": 0.0,
    }
    # Karşılaştırma tablosunun ilk satırı Step 3 gatekeeper RF sonucudur.

    best_model_comparison_row = {
        "Dataset": dataset_key,
        "Comparison": "Best 12-model result",
        "ModelType": best_model["ModelType"],
        "FeatureSet": best_model["FeatureSet"],
        "SelectionMethod": best_model["SelectionMethod"],
        "K": int(best_model["K"]),
        "FeatureCount": int(best_model["FeatureCount"]),
        "ROC": best_model["ROC"],
        "AP": best_model["AP"],
        "F1": best_model["F1"],
        "ROC_Delta_vs_Gatekeeper": best_model["ROC"] - gatekeeper_row["ROC"],
        "ROC_Gain_%": 100 * (best_model["ROC"] - gatekeeper_row["ROC"]) / abs(gatekeeper_row["ROC"]) if abs(gatekeeper_row["ROC"]) > 1e-12 else np.nan,
        "AP_Delta_vs_Gatekeeper": best_model["AP"] - gatekeeper_row["AP"],
        "AP_Gain_%": 100 * (best_model["AP"] - gatekeeper_row["AP"]) / abs(gatekeeper_row["AP"]) if abs(gatekeeper_row["AP"]) > 1e-12 else np.nan,
        "F1_Delta_vs_Gatekeeper": best_model["F1"] - gatekeeper_row["F1"],
        "F1_Gain_%": 100 * (best_model["F1"] - gatekeeper_row["F1"]) / abs(gatekeeper_row["F1"]) if abs(gatekeeper_row["F1"]) > 1e-12 else np.nan,
    }
    # Karşılaştırma tablosunun ikinci satırı 12 model içindeki ROC-AUC temelli en iyi sonuçtur.

    comparison_rows.extend([reference_row, best_model_comparison_row])
    # İki satırlık karşılaştırma genel listeye eklenir.

all_models_df = pd.DataFrame(all_model_rows)
# Tüm model sonuçları tabloya çevrilir.
best_model_df = pd.DataFrame(best_model_rows)
# En iyi model sonuçları tabloya çevrilir.
next_step_model_df = pd.concat(next_step_model_rows, ignore_index=True)
# Sonraki adıma taşınacak RandomForest sonuçları tek tabloda birleştirilir.
comparison_df = pd.DataFrame(comparison_rows)
# Gatekeeper vs en iyi model karşılaştırması tabloya çevrilir.

all_models_df.to_csv(lesson_out / "05_12_model_metrics_all.csv", index=False)
# Tüm model sonuçları CSV olarak kaydedilir.
best_model_df.to_csv(lesson_out / "05_best_model_per_dataset.csv", index=False)
# En iyi model sonuçları CSV olarak kaydedilir.
next_step_model_df.to_csv(lesson_out / "05_next_step_random_forest_only.csv", index=False)
# Sonraki adıma taşınacak RandomForest sonuçları CSV olarak kaydedilir.
comparison_df.to_csv(lesson_out / "05_gatekeeper_vs_best_12_model_all.csv", index=False)
# Gatekeeper ve en iyi 12-model sonucu CSV olarak kaydedilir.

gain_df = add_gain(best_model_df, gatekeeper_df, "05_10_Model_Search", "03_Gatekeeper_Feature_Set")
# En iyi model sonucu Step 3 gatekeeper sonucu ile karşılaştırılır.
gain_df.to_csv(lesson_out / "05_gain_vs_gatekeeper.csv", index=False)
# Kazanç tablosu CSV olarak kaydedilir.

for dataset_key in DATASETS:
    # İki classification veri seti için ayrı gatekeeper vs en iyi model tablosu üretilir.
    dataset_comparison_table = comparison_df[comparison_df["Dataset"] == dataset_key].copy()
    # İlgili veri setinin iki satırlık karşılaştırma tablosu alınır.
    dataset_comparison_table.to_csv(lesson_out / f"{dataset_key}_gatekeeper_vs_best_12_model.csv", index=False)
    # Veri setine özel karşılaştırma tablosu CSV olarak kaydedilir.

    show_table(
        dataset_comparison_table[
            [
                "Dataset",
                "Comparison",
                "ModelType",
                "FeatureSet",
                "SelectionMethod",
                "K",
                "FeatureCount",
                "ROC",
                "AP",
                "F1",
                "ROC_Delta_vs_Gatekeeper",
                "ROC_Gain_%",
                "AP_Delta_vs_Gatekeeper",
                "AP_Gain_%",
                "F1_Delta_vs_Gatekeeper",
                "F1_Gain_%",
            ]
        ],
        title=f"{dataset_key} — Gatekeeper RF vs en iyi 12-model sonucu"
    )
    # Her veri seti için gatekeeper ve en iyi model karşılaştırması ekranda gösterilir.

show_table(
    gain_df[["Dataset", "ModelType", "ROC", "Previous_ROC", "ROC_Delta", "ROC_Gain_%", "AP", "F1"]],
    title="05 — Step 3 gatekeeper sonucuna göre kazanç"
)
# Gatekeeper sonucuna göre performans kazancı ekranda gösterilir.

show_table(
    next_step_model_df[["Dataset", "NextStepDecision", "ModelType", "FeatureSet", "FeatureCount", "ROC", "AP", "F1"]],
    title="Sonraki adıma taşınacak model: RandomForest"
)
# Sampling ve sonraki iyileştirme adımlarına yalnızca RandomForest modeli taşınır.
